In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, FloatSlider, IntSlider, HBox, Layout, VBox, HTML
from IPython.display import display

stats_html = HTML()

def fixed_point_binary(value, integer_bits, fractional_bits):
    total_bits = 1 + integer_bits + fractional_bits
    scale = 2**fractional_bits
    integer_value = int(np.round(value * scale))

    if integer_value < 0:
        integer_value = (1 << total_bits) + integer_value

    binary_string = format(integer_value, f'0{total_bits}b')

    sign = binary_string[0]
    integer_part = binary_string[1:1 + integer_bits]
    fractional_part = binary_string[1 + integer_bits:]

    return sign, integer_part, fractional_part


def plot_fixed_point(x=0.37, integer_bits=1, fractional_bits=4):
    delta = 2.0**(-fractional_bits)

    x_min = -(2.0**integer_bits)
    x_max = 2.0**integer_bits - delta

    x_clipped = np.clip(x, x_min, x_max)
    x_quantized = np.round(x_clipped / delta) * delta
    x_quantized = np.clip(x_quantized, x_min, x_max)

    error = x - x_quantized

    sign, integer_part, fractional_part = fixed_point_binary(x_quantized, integer_bits, fractional_bits)

    levels = np.arange(x_min, x_max + delta / 2, delta)

    fig, ax = plt.subplots(figsize=(9, 4.5))

    ax.scatter(levels, np.zeros_like(levels), s=18)

    ax.axvline(x, color='red', linewidth=2, label='Original value')

    ax.axvline(x_quantized, color='black', linewidth=2, linestyle='--', label='Represented value')

    ax.set_xlim(x_min - 0.1 * (x_max - x_min), x_max + 0.1 * (x_max - x_min))

    ax.set_ylim(-0.5, 0.5)

    ax.set_yticks([])

    ax.set_xlabel('Numerical value', fontsize=13)

    ax.grid(True, axis='x', linestyle=':', alpha=0.6)

    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), borderaxespad=0)

    plt.title('Fixed-Point Representation and Quantization', fontsize=13, pad=15)

    plt.show()

    stats_html.value = f"""
    <div style="font-family: monospace; font-size: 14px; line-height: 1.7;">
    <b>Q format:</b> Q{integer_bits}.{fractional_bits}<br>
    <b>Total word length:</b> {1 + integer_bits + fractional_bits} bits<br>
    <b>Representable range:</b> [{x_min:.6f}, {x_max:.6f}]<br>
    <b>Resolution Δ:</b> {delta:.6f}<br>
    <b>Original value:</b> {x:.6f}<br>
    <b>Represented value:</b> {x_quantized:.6f}<br>
    <b>Quantization error:</b> {error:.6f}<br>
    <b>Binary representation:</b> {sign} {integer_part}.{fractional_part}
    </div>
    """


slider_layout = Layout(width='255px')

style_opts = {'description_width': '78px'}

x_slider = FloatSlider(min=-1.9, max=1.9, step=0.01, value=0.37, description='Value x:', style=style_opts, layout=slider_layout, continuous_update=True)

integer_slider = IntSlider(min=0, max=3, step=1, value=1, description='Integer bits:', style=style_opts, layout=slider_layout, continuous_update=True)

fractional_slider = IntSlider(min=1, max=8, step=1, value=4, description='Fraction bits:', style=style_opts, layout=slider_layout, continuous_update=True)

widget_plot = interactive(plot_fixed_point, x=x_slider, integer_bits=integer_slider, fractional_bits=fractional_slider)

theory_html = HTML("""
<div style="font-family: monospace; font-size: 13px; line-height: 1.55; margin-bottom: 10px;">

<b>Integer bits:</b> Control the dynamic range of the fixed-point representation.<br>
Increasing the number of integer bits extends the range of representable values.<br><br>

<b>Fraction bits:</b> Control the resolution of the representation.<br>
Increasing the number of fraction bits decreases the quantization step Δ, increases the resolution, and generally reduces the quantization error.<br><br>

<b>Value x:</b> Changes the number to be represented. The dashed line shows the nearest available fixed-point value.<br>
As x moves between two adjacent representable levels, its stored value remains fixed until the nearest quantization level changes.<br><br>

<b>Blue dots:</b> Representable fixed-point values.<br>
<b>Red line:</b> Original numerical value.<br>
<b>Dashed black line:</b> Quantized representable value.

</div>
""")

stats_html.layout = Layout(width='375px', min_width='375px')

controls = VBox(
    [x_slider, integer_slider, fractional_slider],
    layout=Layout(
        width='260px',
        min_width='260px',
        overflow='visible',
        justify_content='flex-start',
        margin='0 0 0 5px'
    )
)

bottom_layout = HBox(
    [stats_html, controls],
    layout=Layout(
        width='640px',
        align_items='flex-start',
        justify_content='flex-start',
        overflow='visible'
    )
)

main_layout = VBox(
    [widget_plot.children[-1], bottom_layout],
    layout=Layout(
        width='900px',
        align_items='flex-start',
        overflow='visible'
    )
)

display(
    VBox(
        [theory_html, main_layout],
        layout=Layout(
            width='100%',
            align_items='flex-start',
            overflow='visible'
        )
    )
)